# Data Preprocessing

This notebook prepares the Recruitment dataset for neural network modeling by selecting features, encoding categorical variables, splitting the dataset, and scaling numerical features while avoiding data leakage.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer

In [2]:
data_path = Path("../data/Datasets.xlsx")

df = pd.read_excel(
    data_path,
    sheet_name="Recruitment"
)

df.head()

,Candidate_ID,Position,Experience,Tech_Score,Interview,Status
0,C0001,DS,4,78,83,Hired
1,C0002,DS,6,99,63,Hired
2,C0003,SE,7,78,69,Rejected
3,C0004,SE,1,89,42,Rejected
4,C0005,QA,4,77,48,Pending


In [3]:
X = df.drop(columns=["Candidate_ID", "Status"])
y = df["Status"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (200, 4)
Target shape: (200,)


In [4]:
X.head()

,Position,Experience,Tech_Score,Interview
0,DS,4,78,83
1,DS,6,99,63
2,SE,7,78,69
3,SE,1,89,42
4,QA,4,77,48


In [5]:
y.head()

0       Hired
1       Hired
2    Rejected
3    Rejected
4     Pending
Name: Status, dtype: str

In [6]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Classes:", label_encoder.classes_)
print("Encoded target sample:", y_encoded[:10])

Classes: ['Hired' 'Pending' 'Rejected']
Encoded target sample: [0 0 2 2 1 0 0 0 1 1]


In [7]:
for index, class_name in enumerate(label_encoder.classes_):
    print(f"{class_name} -> {index}")

Hired -> 0
Pending -> 1
Rejected -> 2


In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=0.30,
    random_state=42,
    stratify=y_encoded
)

In [9]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [10]:
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (140, 4)
Validation: (30, 4)
Testing: (30, 4)


In [11]:
print("Training class counts:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nValidation class counts:")
print(pd.Series(y_val).value_counts().sort_index())

print("\nTest class counts:")
print(pd.Series(y_test).value_counts().sort_index())

Training class counts:
0    44
1    48
2    48
Name: count, dtype: int64

Validation class counts:
0     9
1    10
2    11
Name: count, dtype: int64

Test class counts:
0    10
1    10
2    10
Name: count, dtype: int64


In [12]:
numerical_features = [
    "Experience",
    "Tech_Score",
    "Interview"
]

categorical_features = [
    "Position"
]

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [14]:
X_train_processed = preprocessor.fit_transform(X_train)

In [15]:
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

In [16]:
print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (140, 6)
Processed validation shape: (30, 6)
Processed test shape: (30, 6)


In [17]:
feature_names = preprocessor.get_feature_names_out()

print(feature_names)

['num__Experience' 'num__Tech_Score' 'num__Interview' 'cat__Position_DS'
 'cat__Position_QA' 'cat__Position_SE']


In [18]:
processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

processed_df.head()

,num__Experience,num__Tech_Score,num__Interview,cat__Position_DS,cat__Position_QA,cat__Position_SE
0,0.146397,1.406523,0.288301,1.0,0.0,0.0
1,-0.200986,1.749976,-0.627719,0.0,1.0,0.0
2,-1.243138,0.089952,1.605079,0.0,1.0,0.0
3,0.146397,1.120312,-1.543739,1.0,0.0,0.0
4,1.535933,1.063070,1.147069,0.0,1.0,0.0


## Preprocessing Summary

- Candidate_ID was excluded because it is an identifier rather than a predictive feature.
- Status was label encoded for multiclass classification.
- The dataset was divided into 70% training, 15% validation, and 15% testing sets.
- Stratified splitting was used to preserve class distributions.
- Numerical features were standardized using StandardScaler.
- Position was one-hot encoded.
- The preprocessing transformer was fitted only on the training data to prevent data leakage.